In [1]:
import numpy as np
import pandas as pd

In [ ]:
provider_contracts_df = pd.read_csv('../../data/Provider contract.csv')
display(provider_contracts_df)

,Provider Name,Contract type,Shift preference,Total shift count,Weekend shift count,PM shift count
0,Miles Collins,IC,PM,6,4,6
1,Xander Mitchell,FT,"MD1, PM",16,4,4
2,Noah Stevens,IC,"MD1, PM",8,8,8
3,Riley Nelson,IC,"MD1, MD2, PM",6,6,6
4,Riley Stevens,IC,"MD1, PM",0,0,0
5,Miles Walker,FT,"MD1, PM",10,2,2
6,Spencer Stevens,FT,"MD1, PM",12,2,2
7,Spencer Adams,FT,MD2,12,4,0
8,Riley Adams,FT,"MD1, PM",14,4,4
9,Noah Brooks,IC,MD2,4,2,0


In [12]:
import re
from typing import List, Dict, Tuple, Set


SHIFT_ALIASES = {
    "MD1": "MD1",
    "MD2": "MD2",
    "PM": "PM",
    # (If you ever feed "AM" in, treat as both MD1 & MD2)
    "AM": "AM",
    "AM ONLY": "AM",
    "PM ONLY": "PM",
}

def normalize_provider(name: str) -> str:
    return re.sub(r"\s+", " ", name.strip())

def normalize_facility(name: str) -> str:
    return name.strip()

def parse_shift_list(s: str) -> Set[str]:
    """
    Normalize and expand shift preference strings like:
      "MD1, PM" -> {"MD1","PM"}
      "MD1, MD2, PM" -> {"MD1","MD2","PM"}
      "AM" -> {"MD1","MD2"}   (treat AM as both day shifts)
    """
    if pd.isna(s):
        return set()
    parts = [p.strip().upper() for p in str(s).split(",")]
    out = set()
    for p in parts:
        p = SHIFT_ALIASES.get(p, p)
        if p == "AM":
            out.update({"MD1", "MD2"})
        elif p in {"MD1","MD2","PM"}:
            out.add(p)
    return out

def read_provider_contracts(contracts_df) -> pd.DataFrame:
    # Normalize provider names
    contracts_df["provider_name"] = contracts_df["Provider Name"].map(normalize_provider)
    # Normalize shift preferences to sets
    contracts_df["shift_set"] = contracts_df["Shift preference"].apply(parse_shift_list)
    # There may be duplicate provider rows → union their shift sets
    pref = (
        contracts_df.groupby("provider_name", as_index=False)
          .agg(shift_set=("shift_set", lambda sets: set().union(*sets)))
    )
    # Optional: keep a canonical contract type (FT over IC if both present)
    def pick_contract(group):
        types = set(group["Contract type"].dropna().str.upper())
        if "FT" in types:
            return "FT"
        if "IC" in types:
            return "IC"
        return group["Contract type"].iloc[0] if len(group) else None

    pref["contract_type"] = (
        contracts_df.groupby("provider_name")
          .apply(pick_contract)
          .reindex(pref["provider_name"])
          .values
    )
    # Keep the raw counts if you want later (union not needed here)
    return pref

In [13]:
display(read_provider_contracts(provider_contracts_df))

/var/folders/0x/9v5q2xwj44327h79d29_npbw0000gn/T/ipykernel_67929/2187258195.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_contract)


,provider_name,shift_set,contract_type
0,Aiden Adams,{MD2},IC
1,Aiden Davis,"{MD2, MD1, PM}",IC
2,Aiden Evans,{MD2},IC
3,Aiden Reed,"{MD1, PM}",IC
4,Aiden Walker,"{MD2, MD1, PM}",IC
5,Cameron Collins,{MD2},FT
6,Cameron Walker,"{MD1, PM}",IC
7,Dylan Nelson,"{MD1, PM}",FT
8,Hayden Reed,"{MD1, PM}",FT
9,Jordan Brooks,{PM},IC


In [ ]:
provider_credentials_df = pd.read_csv('../../data/Provider Credentialing.csv')
display(provider_credentials_df)

,Provider,Credentialed Facilities,Unnamed: 2
0,Xander Mitchell,"RMC, WREC, WRMC, Wraleigh, Wcary, wplex, Wgarn...",NaN
1,Riley Nelson,"RMC, WREC, WRMC, Wraleigh, Wcary, wplex, Wgarn...",NaN
2,Spencer Stevens,"RMC, WREC, WRMC, Wraleigh, Wcary, wplex, Wgarn...",NaN
3,Riley Adams,"RMC, Wraleigh, Wcary, wplex, Wgarner, Wbrier, ...",NaN
4,Miles Collins,"RMC, WREC, WRMC, Wraleigh, Wcary, wplex, Wgarn...",NaN
5,Noah Stevens,"WREC, WRMC, Wraleigh, Wcary, wplex, Wgarner, W...",NaN
6,Riley Stevens,"RMC, WREC, WRMC, Wraleigh, Wcary, wplex, Wgarn...",NaN
7,Miles Walker,"RMC, WREC, WRMC, Wraleigh, Wcary, wplex, Wgarn...",NaN
8,Spencer Adams,"RMC, Hfax, Hbal, Smain, HM, HB, FMain, FD, FF,...",NaN
9,Noah Brooks,"Hfax, Smain, HM, HB, FMain, FD, FF, FM, SD, ST...",NaN


In [16]:
def read_provider_credentials(credentials_df) -> pd.DataFrame:
    credentials_df.columns = [c.strip() for c in credentials_df.columns]
    credentials_df["Provider"] = credentials_df["Provider"].map(normalize_provider)
    # Some rows may be empty at the end
    credentials_df = credentials_df.dropna(subset=["Provider"]).copy()

    def split_facilities(s: str) -> Set[str]:
        if pd.isna(s) or not str(s).strip():
            return set()
        # split by comma, strip whitespace
        return {normalize_facility(x) for x in str(s).split(",")}
    credentials_df["facility_set"] = credentials_df["Credentialed Facilities"].apply(split_facilities)

    # Deduplicate by union of facility sets across repeated providers
    cred = (
        credentials_df.groupby("Provider", as_index=False)
          .agg(facility_set=("facility_set", lambda sets: set().union(*sets)))
    )
    return cred

In [17]:
display(read_provider_credentials(provider_contracts_df))

KeyError: 'Provider'